# EnHi-LMP · full training in Google Colab
Runs **every experiment and every model** of the project with one script (`scripts/train_all.py`):

| Step | What | Time on a T4 GPU |
|---|---|---|
| 1 | E1–E5: chrF, embedding cosine, token-F1, LegalDiff-GBR, ML-JudgeBERT, ML-JudgeBERT + LegalDiff (640 pairs, grouped 5-fold CV, 3 seeds) | ~20–30 min |
| 2 | Pretrained encoders: XLM-R base, MuRIL base, mBERT (fine-tuned, + LegalDiff late fusion) | ~15 min each |
| 3 | Deployed model + browser model + web app rebuilt and checked | ~1 min |
| 4 | One comparison table: `results/ALL_RESULTS.md` | – |

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all**.
If Colab disconnects, run the cells again: the training cell uses `--resume` and skips finished steps (results are copied to Google Drive after every run if you mount it).

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2. (Recommended) Keep results safe in Google Drive in case Colab disconnects
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 3. Get the code and install everything (~2 min)
import os
%cd /content
if not os.path.exists("NLP_RESERACH_WORK"):
    !git clone -q https://github.com/udbhav968-creator/NLP_RESERACH_WORK.git
%cd /content/NLP_RESERACH_WORK
!git pull -q
!git log --oneline -1
!pip -q install -r requirements.txt -r api/requirements.txt transformers accelerate sentencepiece

In [ ]:
# 4. (Optional) 10-minute smoke test of the whole pipeline. It does not change the real results.
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    !python scripts/train_all.py --quick

In [ ]:
# 5. FULL TRAINING (about 60-80 min on a T4). Safe to re-run: finished steps are skipped.
!python scripts/train_all.py --resume

In [ ]:
# 6. Copy results to Google Drive (if mounted)
import shutil, time, os
if os.path.exists('/content/drive/MyDrive'):
    dst = '/content/drive/MyDrive/EnHi-LMP_results_' + time.strftime('%Y%m%d_%H%M')
    shutil.copytree('results', dst); print('Saved to', dst)

In [ ]:
# 7. All results in one table
from IPython.display import Markdown, display
display(Markdown(open('results/ALL_RESULTS.md', encoding='utf-8').read()))

In [ ]:
# 8. Chart: Pearson r of every model, English vs Hindi
import pandas as pd, matplotlib.pyplot as plt, numpy as np
df = pd.read_csv('results/all_models.csv').sort_values('pearson')
y = np.arange(len(df)); h = .38
fig, ax = plt.subplots(figsize=(9, .45 * len(df) + 1.4))
ax.barh(y - h/2, df.r_en, h, label='English'); ax.barh(y + h/2, df.r_hi, h, label='Hindi')
ax.set_yticks(y); ax.set_yticklabels(df.model, fontsize=8); ax.axvline(0, color='grey', lw=.6)
ax.set_xlabel('Pearson r with LMP labels (640 pairs, grouped 5-fold CV)'); ax.legend(loc='lower right')
ax.set_title('EnHi-LMP: every model, by language'); plt.tight_layout()
plt.savefig('results/fig_all_models_by_language.png', dpi=200); plt.show()

In [ ]:
# 9. Download everything as one zip
!zip -qr enhi_lmp_full_results.zip results web/model.json web/stats640.json
from google.colab import files
files.download('enhi_lmp_full_results.zip')

## 10. (Optional) Push the new results straight to GitHub
Needs a GitHub token with **repo** access (github.com → Settings → Developer settings → Personal access tokens). The token is typed into a hidden box and not saved.

In [ ]:
PUSH_TO_GITHUB = False
if PUSH_TO_GITHUB:
    from getpass import getpass
    token = getpass('GitHub token: ')
    !git config user.email "colab@users.noreply.github.com" && git config user.name "Colab training run"
    !git add results web/model.json web/legal-meaning-checker.html api/static/index.html
    !git commit -q -m "Full training run in Colab (GPU): E1-E5 + XLM-R, MuRIL, mBERT"
    !git push -q https://{token}@github.com/udbhav968-creator/NLP_RESERACH_WORK.git HEAD:main && echo pushed
    del token